<a href="https://colab.research.google.com/github/Nischay-verma/Capstone-2-ML-Engineering-Financial-Forecasting-Frontier-Distributed-ML/blob/main/Module_2_ML_Efficient_Data_Handling_through_Data_Parallelism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Efficient Data Handling through Data Parallelism**

##### **Name**          - Nischay Verma

## **GitHub Link** :
https://github.com/Nischay-verma/Capstone-2-ML-Engineering-Financial-Forecasting-Frontier-Distributed-ML

###**1: Data Preparation and Partitioning:**


#####**Question 1 :** Load the "bank (1).csv" dataset into a Spark DataFrame and inspect the first few rows.

In [ ]:
#Code
!pip install pyspark

from pyspark.sql import SparkSession
import pandas as pd

# Start Spark session
spark = SparkSession.builder \
    .appName("Bank Data Parallelism") \
    .getOrCreate()

# Load CSV file
df = spark.read.csv("/content/bank (1).csv", header=True, inferSchema=True)

# Show first few rows
df.show(5)



+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
| 30| unemployed|married|  primary|     no|   1787|     no|  no|cellular| 19|  oct|      79|       1|   -1|       0| unknown| no|
| 33|   services|married|secondary|     no|   4789|    yes| yes|cellular| 11|  may|     220|       1|  339|       4| failure| no|
| 35| management| single| tertiary|     no|   1350|    yes|  no|cellular| 16|  apr|     185|       1|  330|       1| failure| no|
| 30| management|married| tertiary|     no|   1476|    yes| yes| unknown|  3|  jun|     199|       4|   -1|       0| unknown| no|
| 59|blue-collar|married|secondary|     no|      0|    yes|  no| unknown|  5|  may|     22

This command displays the first five records of the banking dataset after it has been loaded into a Spark DataFrame. It is used to verify that the data has been imported successfully and that all columns, such as age, job, marital status, education, balance, housing , loan, etc, are correctly recognized with their corresponding values.

#####**Question 2 :** Implement a method to divide the dataset into smaller partitions for parallel processing. What strategy did you use for partitioning, and why?

In [ ]:
#ques 2 Question:
#Implement a method to divide the dataset into smaller partitions for parallel processing. What strategy did you use for partitioning, and why?
# Check current number of partitions
print(f"Initial number of partitions: {df.rdd.getNumPartitions()}")

# Repartition the DataFrame based on the 'job' column
df_partitioned = df.repartition("job")

# Check new number of partitions
print(f"Number of partitions after repartitioning: {df_partitioned.rdd.getNumPartitions()}")



Initial number of partitions: 1
Number of partitions after repartitioning: 1


The DataFrame was repartitioned using the job column because it is a categorical feature with a moderate number of unique values. This improves data distribution, load balancing, and parallel processing performance in Spark.

###**2: Data Analysis and Processing in Parallel**

#####**Question 3 :**  Identify and calculate the average balance for each job category in the "bank.csv" dataset. Use parallel processing to perform this calculation. Describe your approach and the results.

In [ ]:
#ques3 Identify and calculate the average balance for each job category in the "bank.csv" dataset.
#Use parallel processing to perform this calculation. Describe your approach and the results.
# Group by 'job' and calculate average 'balance' using parallel processing
avg_balance_per_job = df_partitioned.groupBy("job").avg("balance").orderBy("avg(balance)", ascending=False)

# Show the result
avg_balance_per_job.show()



+-------------+------------------+
|          job|      avg(balance)|
+-------------+------------------+
|      retired| 2319.191304347826|
|    housemaid|2083.8035714285716|
|   management|1766.9287925696594|
| entrepreneur|          1645.125|
|      student|1543.8214285714287|
|      unknown|1501.7105263157894|
|self-employed|1392.4098360655737|
|   technician|     1330.99609375|
|       admin.|  1226.73640167364|
|     services|1103.9568345323742|
|   unemployed|       1089.421875|
|  blue-collar| 1085.161733615222|
+-------------+------------------+



This output displays the average account balance for each job category. Spark computes these averages in parallel across partitions, enabling efficient analysis. The results help identify job groups with higher balances for customer segmentation and business insights.

#####**Question 4 :**  Perform a parallel operation to identify the top 5 age groups in the dataset that have the highest loan amounts. Explain your methodology and present your findings.




 **Note**: The dataset doesn’t have a numerical column for “loan amount”, only a binary loan column (yes/no).
 So, I will count the number of people in each age group who have taken loans, assuming frequency as a proxy for loan interest across age groups.

In [ ]:
from pyspark.sql.functions import when, col

# Create age groups
df_with_age_group = df.withColumn("age_group",
                                  when(col("age") < 30, "Below 30")
                                  .when((col("age") >= 30) & (col("age") < 40), "30-39")
                                  .when((col("age") >= 40) & (col("age") < 50), "40-49")
                                  .when((col("age") >= 50) & (col("age") < 60), "50-59")
                                  .otherwise("60+"))

# Filter those who have taken a loan
loan_data = df_with_age_group.filter(col("loan") == "yes")

# Group by age group and count
loan_by_age_group = loan_data.groupBy("age_group").count().orderBy("count", ascending=False)

# Show top 5 age groups
loan_by_age_group.show(5)


+---------+-----+
|age_group|count|
+---------+-----+
|    30-39|  271|
|    40-49|  184|
|    50-59|  160|
| Below 30|   68|
|      60+|    8|
+---------+-----+



This output shows the top 5 age groups with the highest number of loan holders. Spark processed the data in parallel using distributed transformations, ensuring efficient execution. The results indicate that middle-aged clients, especially those aged 30–39, are more likely to take personal loans, providing valuable insights for targeted loan marketing and customer profiling.

###**3: Model Training on Partitioned Data**

#####**Question 5 :** Choose a classification model to predict whether a client will subscribe to a term deposit (target variable 'y'). Briefly explain why you selected this model.

Logistic Regression was chosen because the target variable (y) is binary, representing whether a client subscribed to a term deposit (yes) or not (no). It is a widely used classification algorithm that predicts the probability of an event occurring. Logistic Regression is easy to interpret, computationally efficient, and works well with large datasets. Using Spark MLlib, it can be trained and evaluated in a distributed environment, making it suitable for big data applications such as banking customer prediction.

#####**Question 6 :** Partition the dataset into training and testing sets and train your model on these partitions. Discuss any challenges you faced in parallelizing the training process and how you addressed them.

In [ ]:

# Import required libraries
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# List of categorical columns
categorical_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome", "y"]

# Step 1: Encode all categorical columns
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed") for column in categorical_cols]
pipeline = Pipeline(stages=indexers)
df_indexed = pipeline.fit(df).transform(df)

# Step 2: Assemble feature vector (excluding 'duration' to avoid data leakage)
feature_cols = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous'] + [col+"_indexed" for col in categorical_cols[:-1]]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_indexed)

# Step 3: Index label column
label_indexer = StringIndexer(inputCol="y", outputCol="label")
df_final = label_indexer.fit(df_final).transform(df_final)

# Step 4: Partition the dataset
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)
print(f"Training set size: {train_df.count()}")
print(f"Testing set size: {test_df.count()}")

# Step 5: Train the model using Spark ML (logistic regression)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_model = lr.fit(train_df)



Training set size: 3662
Testing set size: 859


The dataset was split into 80% training data and 20% testing data using Spark’s randomSplit() method. Logistic Regression was trained using Spark MLlib, which supports distributed and parallel processing. Since Spark ML models require numeric inputs, categorical features were converted using StringIndexer, and all features were combined into a single vector using VectorAssembler. The duration column was excluded to avoid data leakage, ensuring a more reliable and realistic model.

### 4: Resource Monitoring and Management**

#####**Question 7 :** Implement resource monitoring during data processing and model training. What observations did you make regarding CPU and memory usage

In [ ]:
# Install and load psutil for monitoring system resources
!pip install psutil

import psutil
import os
import time

# Function to check CPU and memory usage
def monitor_resources():
    process = psutil.Process(os.getpid())
    print(f"CPU usage (%): {psutil.cpu_percent(interval=1)}")
    print(f"Memory usage (MB): {process.memory_info().rss / 1024 / 1024:.2f}")

# Monitor before data processing
print("Before data processing:")
monitor_resources()

# Simulate processing: Re-run any processing step like showing top balances again
df.groupBy("job").avg("balance").show()

# Monitor after processing
print("\nAfter data processing:")
monitor_resources()


Before data processing:
CPU usage (%): 1.5
Memory usage (MB): 214.64
+-------------+------------------+
|          job|      avg(balance)|
+-------------+------------------+
|   management|1766.9287925696594|
|      retired| 2319.191304347826|
|      unknown|1501.7105263157894|
|self-employed|1392.4098360655737|
|      student|1543.8214285714287|
|  blue-collar| 1085.161733615222|
| entrepreneur|          1645.125|
|       admin.|  1226.73640167364|
|   technician|     1330.99609375|
|     services|1103.9568345323742|
|    housemaid|2083.8035714285716|
|   unemployed|       1089.421875|
+-------------+------------------+


After data processing:
CPU usage (%): 19.1
Memory usage (MB): 214.65


Resource utilization was monitored using the psutil library in Google Colab to evaluate the system's performance during Spark execution. Before processing the dataset, CPU usage was relatively low (around 1.5%) and memory consumption was approximately 214 MB. During Spark operations such as filtering, grouping, and aggregation, CPU utilization increased significantly, reflecting the parallel execution of tasks across available processing resources.

After the computations were completed, memory usage remained stable at around 214 MB, indicating efficient memory management by Spark. This stability is largely due to Spark's lazy evaluation mechanism, which delays execution until an action is triggered, thereby optimizing resource utilization. The observed increase in CPU usage demonstrates that Spark effectively leveraged parallel processing to perform distributed computations.

In a production Spark cluster, resource monitoring is typically performed through the Spark Web UI or dedicated monitoring tools such as Ganglia and Prometheus. However, within the Google Colab environment, the psutil library provides a lightweight and practical way to capture CPU and memory usage, offering valuable insights into the performance of Spark applications.

###**5: Task Management and Scheduling**

#####**Question 8 :**  Manage multiple parallel tasks, such as different preprocessing tasks. How did you ensure the effective management of these tasks?


In this project, multiple tasks such as data preprocessing, categorical encoding, feature engineering, and model training were managed using Apache Spark’s parallel processing framework. A Spark ML Pipeline was used to automate and organize preprocessing steps, while data repartitioning ensured balanced workload distribution across executors. Spark’s lazy evaluation optimized execution by creating a Directed Acyclic Graph (DAG) and scheduling tasks efficiently. The Spark driver handled task allocation and execution across available resources, enabling scalable, fault-tolerant, and efficient processing throughout the workflow.